# Thêm Thư Viện

In [2]:
import pyodbc
import pandas as pd

# Tạo kết nối

In [5]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_lib = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_lib;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)


# ETL bảng Dim_Date

## Xóa data bảng DIM_date 

In [51]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Date"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ file CSV

In [ ]:
df_data_date = pd.read_csv("./Source-Data/Dim_Date.csv")
print(df_data_date)

       Date_key   Full_date                     Date_text   Day_name  \
0      19700101    1/1/1970     Thursday, January 1, 1970   Thursday   
1      19700102    1/2/1970       Friday, January 2, 1970     Friday   
2      19700103    1/3/1970     Saturday, January 3, 1970   Saturday   
3      19700104    1/4/1970       Sunday, January 4, 1970     Sunday   
4      19700105    1/5/1970       Monday, January 5, 1970     Monday   
...         ...         ...                           ...        ...   
29215  20491227  12/27/2049     Monday, December 27, 2049     Monday   
29216  20491228  12/28/2049    Tuesday, December 28, 2049    Tuesday   
29217  20491229  12/29/2049  Wednesday, December 29, 2049  Wednesday   
29218  20491230  12/30/2049   Thursday, December 30, 2049   Thursday   
29219  20491231  12/31/2049     Friday, December 31, 2049     Friday   

       Week_of_quarter  Day_of_week  Month  Quarter  Year  Day  
0                    1            4      1        1  1970    1  
1    

## Load data vào dwh_lib

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Date (Date_key, Full_date, Date_text, Day, Week_of_quarter, Month, Quarter, Year, Day_of_week, Day_name)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """
for index, row in df_data_date.iterrows():
    values = (row['Date_key'], 
              row['Full_date'], 
              row['Date_text'], 
              row['Day'], 
              row['Week_of_quarter'], 
              row['Month'], 
              row['Quarter'], 
              row['Year'], 
              row['Day_of_week'], 
              row['Day_name'])
    cursor_dwh.execute(insert_query, values)
    conn_dwh_lib.commit()

# ETL bảng Dim_Khoa

## Xóa data bảng DIM_Khoa

In [52]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Khoa"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

IntegrityError: ('23000', '[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The DELETE statement conflicted with the REFERENCE constraint "FK_Lop_Khoa". The conflict occurred in database "dwh_lib", table "dbo.DIM_Lop", column \'ID_khoa\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)')

## Đọc data từ CSV

In [48]:
df_data_khoa = pd.read_csv("./Source-Data/Data_Dim_Khoa.csv")
df_data_khoa['Ten_khoa'] = df_data_khoa['Ten_khoa'].apply(lambda x: x.title() if isinstance(x, str) else x)

print(df_data_khoa)

    ID_Khoa                             Ten_khoa
0         0                             Không Rõ
1         1                    Lý Luận Chính Trị
2         2                    Khoa Học Ứng Dụng
3         3                   Cơ Khí Chế Tạo Máy
4         4                       Điện - Điện Tử
5         5                      Cơ Khí Động Lực
6         6                              Kinh Tế
7         7                  Công Nghệ Thông Tin
8         8                   In Và Truyền Thông
9         9          Công Nghệ May Và Thời Trang
10       10       Công Nghệ Hóa Học Và Thực Phẩm
11       11                             Xây Dựng
12       12                            Ngoại Ngữ
13       13               Đào Tạo Chất Lượng Cao
14       14                Viện Sư Phạm Kỹ Thuật
15       15  Trường Trung Học Kỹ Thuật Thực Hành


## Load data vào bảng Dim_Khoa

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Khoa (ID_khoa, Ten_khoa) 
                VALUES (?, ?)
                """
for index, row in df_data_khoa.iterrows():
    values = (row['ID_Khoa'], 
              row['Ten_khoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Dan_Toc

## Xóa data bảng DIM_Dan_Toc

In [167]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Dan_toc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ CSV

In [ ]:
df_data_dantoc = pd.read_csv("./Source-Data/Data_Dim_Dan_toc.csv")
df_data_dantoc = df_data_dantoc.where(pd.notnull(df_data_dantoc), None)
print(df_data_dantoc)

    Mã               Tên                                       Tên gọi khác
0    1              Kinh                                               Việt
1    2               Tày          Thổ, Ngạn, Phén, Thù Lao, Pa Dí, Tày Khao
2    3              Thái  Tày Đăm, Tày Mười, Tày Thanh, Mán Thanh, Hàng ...
3    4               Hoa  Hán, Triều Châu, Phúc Kiến, Quảng Đông, Hải Na...
4    5            Khơ-me             Cur, Cul, Cu, Thổ, Việt gốc Miên, Krôm
5    6             Mường               Mol, Mual, Mọi, Mọi Bi, Ao Tá, Ậu Tá
6    7              Nùng  Xuồng, Giang, Nùng An, Phàn Sinh, Nùng Cháo, N...
7    8             HMông  Mèo, Hoa, Mèo Xanh, Mèo Đỏ, Mèo Đen, Ná Mẻo, M...
8    9               Dao  Mán, Động, Trại, Xá, Dìu, Miên, Kiềm, Miền, Qu...
9   10           Gia-rai    Giơ-rai, Tơ-buăn, Chơ-rai, Hơ-bau, Hđrung, Chor
10  11              Ngái                            Xín, Lê, Đản, Khách Gia
11  12              Ê-đê  Ra-đê, Đê, Kpạ, A-đham, Krung, Ktul, Đliê Ruê,...
12  13      

## Load data vào bảng DIM_Dan_toc

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Dan_toc (ID_dan_toc, Dan_toc, Ten_khac) 
                VALUES (?, ?, ?)
                """
for index, row in df_data_dantoc.iterrows():
    values = (row['Mã'], 
              row['Tên'], 
              row['Tên gọi khác'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Trinh_do

## Xóa data bảng Dim_Trinh_do

In [211]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Trinh_do"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [ ]:
query_trinhdo = "SELECT trinh_do_id, dbo.DecodeUTF8String(loai_trinh_do) AS Trinh_do FROM Trinh_do "
df_trinhdo = pd.read_sql(query_trinhdo, conn_libol)
print(df_trinhdo)

   trinh_do_id                 Trinh_do
0           10                   PGS.TS
1            3                 Cao đẳng
2            4                  Đại học
3            5                  Thạc sĩ
4            6                  Tiến sĩ
5            7              Phó tiến sĩ
6           11  Trung học chuyên nghiệp
7            9             Trung học PT
8           12                    12/12


C:\Users\admin\AppData\Local\Temp\ipykernel_17252\57140960.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Trinh_do = pd.read_sql(query_Trinh_do, conn_libol)


## Load data vào bảng Dim

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Trinh_do (ID_trinh_do, Loai_trinh_do) 
                VALUES (?, ?)
                """
for index, row in df_trinhdo.iterrows():
    values = (row['trinh_do_id'], 
              row['Trinh_do'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng DIM_Khoa_hoc

## Xóa data bảng DIM_Khoa_hoc

In [32]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Khoa_hoc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ CSV

In [33]:
df_data_khoahoc = pd.read_csv("./Source-Data/Data_Dim_Khoa_hoc.csv")
print(df_data_khoahoc)

   ID khóa học    Tên khóa học
0           70       1970-1974
1           71       1971-1975
2            0  Không xác định
3           73       1973-1977
4           74       1974-1978
..         ...             ...
76         K46       2046-2050
77         K47       2047-2051
78         K48       2048-2052
79         K49       2049-2053
80         K50       2050-2054

[81 rows x 2 columns]


## Load data vào bảng Dim_Khoa

In [34]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Khoa_hoc (ID_khoa_hoc, Ten_khoa_hoc) 
                VALUES (?, ?)
                """
for index, row in df_data_khoahoc.iterrows():
    values = (row['ID khóa học'], 
              row['Tên khóa học'])
    cursor_dwh.execute(insert_query, values)   
conn_dwh_lib.commit()

# ETL bảng Dim_Lop

## Xóa data bảng Dim_Lop

In [61]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Lop"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [62]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Lop = "SELECT Ten_lop FROM Lop"
df_lop_lop = pd.read_sql(query_Lop, conn_libol)

# Đọc dữ liệu từ bảng Ban_doc trong CSDL libol
query_LopBandoc = "SELECT DISTINCT dbo.DecodeUTF8String(Lop) AS Lop FROM Ban_doc"
df_lop_bandoc = pd.read_sql(query_LopBandoc, conn_libol)

print(df_lop_lop)
print(df_lop_bandoc)

C:\Users\phung\AppData\Local\Temp\ipykernel_27428\869524976.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop_lop = pd.read_sql(query_Lop, conn_libol)
C:\Users\phung\AppData\Local\Temp\ipykernel_27428\869524976.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop_bandoc = pd.read_sql(query_LopBandoc, conn_libol)


       Ten_lop
0      191040B
1      191040A
2    19109CL1B
3    19109CL1A
4    19109CL2B
..         ...
597    211611B
598    211611A
599    211612A
600    211612B
601      21950

[602 rows x 1 columns]
            Lop
0        017031
1       001011C
2       042030A
3       057090A
4       117450A
...         ...
4259        709
4260   14151CLC
4261    181311A
4262  19143CL1B
4263      23950

[4264 rows x 1 columns]


## Xử lý data

In [63]:
# Tạo data_frame mới gộp các hàng dữ liệu từ 2 data_frame kia
df_data_lop = pd.DataFrame({"Ten_lop": pd.concat([df_lop_lop["Ten_lop"], 
                                                  df_lop_bandoc["Lop"]], 
                                                  ignore_index=True)})
df_data_lop['Ten_lop'] = df_data_lop['Ten_lop'].str.upper() # In hoa hết các hàng dữ liệu
for j, row in df_data_lop.iterrows():
    ten_nhom = row["Ten_lop"]
    if ((pd.isna(ten_nhom)) or # Kiểm tra none
        (ten_nhom == "") or
        (ten_nhom == "0") or
        (ten_nhom == "00") or
        (ten_nhom == "000")):  # Kiểm tra NaN
        df_data_lop.at[j, 'Ten_lop'] = "Không rõ"

df_data_lop = df_data_lop.sort_values(by="Ten_lop", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
df_data_lop = df_data_lop.drop_duplicates().reset_index(drop=True) # xóa những hàng bị trùng nhau
df_data_lop['ID_khoa'] = "0" # cho ID_Khoa = 0 (Không rõ) vì hiện tại chưa có dữ liệu về lớp thuộc khoa nào 
print(df_data_lop)

       Ten_lop ID_khoa
0       001011       0
1      001011A       0
2      001011C       0
3       001012       0
4       001013       0
...        ...     ...
4256  XÂY DỰNG       0
4257   ÊN11021       0
4258  ÊN14010A       0
4259  ÊN2D02VD       0
4260      ĐIỆN       0

[4261 rows x 2 columns]


## Load data vào bảng Dim_Lop

In [66]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Lop (ID_lop, ID_khoa) 
                VALUES (?, ?)
                """
for index, row in df_data_lop.iterrows():
    values = (row['Ten_lop'], 
              row['ID_khoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

IntegrityError: ('23000', "[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot insert the value NULL into column 'Ten_lop', table 'dwh_lib.dbo.DIM_Lop'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)")

# ETL bảng Dim_Nhom_ban_doc

## Xóa data bảng Dim_Nhom_ban_doc

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Nhom_ban_doc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [27]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Nhombandoc = "SELECT Nhom_ID, dbo.DecodeUTF8String(Ten_nhom) AS Ten_nhom FROM Nhom_ban_doc"
df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_libol)
print(df_nhombandoc)

    Nhom_ID                  Ten_nhom
0         5                          
1         6         .Cán bộ công chức
2         9          MƯỢN & ĐỌC - SKV
3        10  Tốt nghiệp_Cộng Tác viên
4        12               Đọc tại chỗ
5        14                   MƯỢN GT
6        15             MƯỢN GT & SKV
7        16    Chưa tham gia khóa học
8        17    Con CB & CTV (Mượn GT)
9        18          HỌC VIÊN CAO HỌC
10       19    Khoa ĐT chất lượng cao
11       20         NHÓM NGOÀI TRƯỜNG
12       21       GIÁO TRÌNH QUÉT LỘN
13       22    NHÓM LÃNH ĐẠO, QUẢN LÝ
14       23         Nhóm ngoài trường
15       24      Nhóm ký công nợ (TN)
16       25           Nghiên cứu sinh


C:\Users\admin\AppData\Local\Temp\ipykernel_21948\1031425255.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_libol)


## Xử lý data

In [59]:
df_nhombandoc = df_nhombandoc.drop_duplicates().reset_index(drop=True) # xóa những hàng bị trùng nhau
for j, row in df_nhombandoc.iterrows():
    ten_nhom = row["Ten_nhom"]
    if pd.isna(ten_nhom) or ten_nhom == "":  # Kiểm tra NaN
        df_nhombandoc.at[j, 'Ten_nhom'] = "Không rõ"
    else:
        ten_nhom = ten_nhom[0].upper() + ten_nhom[1:].lower() # Chỉnh sửa chữ hoa chữ thường nếu chuỗi không rỗng
        df_nhombandoc.at[j, 'Ten_nhom'] = ten_nhom

print(df_nhombandoc)


    Nhom_ID                  Ten_nhom
0         5                  Không rõ
1         6         .cán bộ công chức
2         9          Mượn & đọc - skv
3        10  Tốt nghiệp_cộng tác viên
4        12               Đọc tại chỗ
5        14                   Mượn gt
6        15             Mượn gt & skv
7        16    Chưa tham gia khóa học
8        17    Con cb & ctv (mượn gt)
9        18          Học viên cao học
10       19    Khoa đt chất lượng cao
11       20         Nhóm ngoài trường
12       21       Giáo trình quét lộn
13       22    Nhóm lãnh đạo, quản lý
14       23         Nhóm ngoài trường
15       24      Nhóm ký công nợ (tn)
16       25           Nghiên cứu sinh


## Load data

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Nhom_ban_doc (ID_nhom_ban_doc, Nhom_ban_doc) 
                VALUES (?, ?)
                """
for index, row in df_nhombandoc.iterrows():
    values = (row['Nhom_ID'], 
              row['Ten_nhom'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Nhom_nghanh_nghe

## Xóa data bảng Dim_Nhom_nghanh_nghe

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Nhom_nghanh_nghe"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [57]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Nhomnghanhnghe = "SELECT ID, dbo.DecodeUTF8String(Ten_nhom) AS Ten_nhom FROM Nhom_nghanh_nghe"
df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_libol)
print(df_nhomnghanhnge)

C:\Users\admin\AppData\Local\Temp\ipykernel_21948\443329784.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_libol)


      ID                  Ten_nhom
0      0          (Không xác định)
1      1                      Y tế
2      5       Công nhân viên chức
3      9         Điện tử - Tin học
4     11                 Tài chính
..   ...                       ...
248  291        Công nghệ vật liệu
249  292         Vật liệu xây dựng
250  293                      Luật
251  294         Sư phạm công nghệ
252  295  Kỹ thuật cơ khí động lực

[253 rows x 2 columns]


## Xử lý data

In [69]:
for j, row in df_nhomnghanhnge.iterrows(): 
    ten_nhom = row["Ten_nhom"]
    if pd.isna(ten_nhom) or ten_nhom == "":  # Kiểm tra none hoặc NaN
        df_nhombandoc.at[j, 'Ten_nhom'] = "(Không xác định)" 
df_nhomnghanhnge = df_nhomnghanhnge.drop_duplicates(subset='Ten_nhom').reset_index(drop=True) # xóa những hàng bị trùng nhau

print(df_nhomnghanhnge)

      ID                          Ten_nhom
0      0                  (Không xác định)
1      1                              Y tế
2      5               Công nhân viên chức
3      9                 Điện tử - Tin học
4     11                         Tài chính
..   ...                               ...
232  288  Logistic và Tài chính thương mại
233  292                 Vật liệu xây dựng
234  293                              Luật
235  294                 Sư phạm công nghệ
236  295          Kỹ thuật cơ khí động lực

[237 rows x 2 columns]


## Load data

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Nhom_nghanh_nghe (ID_nhom_nghanh_nghe, 
                            Nhom_nghanh_nghe) 
                VALUES (?, ?)
                """
for index, row in df_nhomnghanhnge.iterrows():
    values = (row['ID'], 
              row['Ten_nhom'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Quoc_gia

## Xóa data bảng Dim_Quoc_gia

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Quoc_gia"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [66]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Quocgia = "SELECT Ma_nuoc_ID, Ma_ISO,Ten_nuoc_ISO FROM Ten_nuoc"
df_quocgia = pd.read_sql(query_Quocgia, conn_libol)
print(df_quocgia)

     Ma_nuoc_ID Ma_ISO         Ten_nuoc_ISO
0           190     TG                 Togo
1           191     TK              Tokelau
2           192     TO                Tonga
3           193     TT  Trinidad and Tobago
4           194     TN              Tunesia
..          ...    ...                  ...
231         186     SY                Syria
232         187     TW               Taiwan
233         188     TZ             Tanzania
234         189     TH             Thailand
235         209     VN              Vietnam

[236 rows x 3 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21948\1487321348.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_quocgia = pd.read_sql(query_Quocgia, conn_libol)


## Xử lý data

In [70]:
df_quocgia = df_quocgia.sort_values(by="Ma_nuoc_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
df_quocgia = df_quocgia.drop_duplicates(subset='Ten_nuoc_ISO').reset_index(drop=True) # xóa những hàng bị trùng nhau

print(df_quocgia)

     Ma_nuoc_ID Ma_ISO        Ten_nuoc_ISO
0             1     AF         Afghanistan
1             2     AL             Albania
2             3     DZ             Algeria
3             4     AS      American Samoa
4             5     AD             Andorra
..          ...    ...                 ...
231         232     AJ          Azerbaijan
232         233     CI             Croatia
233         234     XV            Slovenia
234         235     BN  Bosnia-Hercegovina
235         236     XN           Macedonia

[236 rows x 3 columns]


## Load data

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Quoc_gia (ID_quoc_gia, Ma_ISO, Ten_nuoc_ISO) 
                VALUES (?, ?, ?)
                """
for index, row in df_quocgia.iterrows():
    values = (row['Ma_nuoc_ID'], 
              row['Ma_ISO'],
              row['Ten_nuoc_ISO'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Vat_mang_tin

## Xóa data bảng 

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Vat_mang_tin"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data SQL Server

In [17]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Vatmangtin = "SELECT Vat_mang_tin_ID, dbo.DecodeUTF8String(Ky_hieu) AS Ky_hieu, dbo.DecodeUTF8String(Vat_mang_tin) AS Vat_mang_tin FROM Vat_mang_tin"
df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_libol)
print(df_vatmangtin)

    Vat_mang_tin_ID Ky_hieu          Vat_mang_tin
0                 1      MC             Microfilm
1                 2      MF            Microfiche
2                 3       G                  Giấy
3                 4      BT               Băng từ
4                 5      ĐT                Đĩa từ
5                 6      CD  Đĩa CDROM (đĩa Laze)
6                 7      VA    Vật liệu nghe nhìn
7                 8    giấy                  None
8                 9  Dia tu                  None
9                10      GH                  None
10               11       B                  None
11               12     Tan                  None
12               13   Ebook          Sách điện tử
13               14      VT                  None
14               15       A                  None


C:\Users\phung\AppData\Local\Temp\ipykernel_24616\2402062624.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_libol)


## Load vào bảng Dim_Vat_mang_tin

In [18]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Vat_mang_tin (ID_vat_mang_tin, Ky_hieu, Vat_mang_tin) 
                VALUES (?, ?, ?)
                """
for index, row in df_vatmangtin.iterrows():
    values = (row['Vat_mang_tin_ID'], 
              row['Ky_hieu'],
              row['Vat_mang_tin'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Dang_tai_lieu

## Xóa data bảng Dim_Dang_tai_lieu

In [73]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Dang_tai_lieu"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data

In [74]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Dangtailieu = """SELECT Dang_tai_lieu_ID, 
                        dbo.DecodeUTF8String(Dang_tai_lieu) AS Dang_tai_lieu, 
                        dbo.DecodeUTF8String(Ky_hieu_tai_lieu) AS Ky_hieu_tai_lieu,
                        LoanPeriod,
                        Renewals,
                        RenewalPeriod,
                        TimeUnit,
                        Fee,
                        OverdueFine,
                        FixedFee
                        FROM Dang_tai_lieu
                        """
df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_libol)
print(df_dangtailieu)

    Dang_tai_lieu_ID                                    Dang_tai_lieu  \
0                  1                     Sách, chuyên khảo, tuyển tập   
1                  2                                        Bài trích   
2                  3                                Luận án, luận văn   
3                  4  Báo cáo kết quả  nghiên cứu; tổng kết; khảo sát   
4                  5                                 Báo cáo hội nghị   
5                  6                               Catalô công nghiệp   
6                  7                                       Tiêu chuẩn   
7                  8                                         Sáng chế   
8                  9                                  ấn phẩm định kỳ   
9                 10                                             Phim   
10                11                              Bản đồ, sách bản đồ   
11                12                    Hình vẽ, bản vẽ, tranh ảnh,..   
12                13                     Tờ rời giớ

C:\Users\phung\AppData\Local\Temp\ipykernel_24616\3235927709.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_libol)


## Xử lý data


In [75]:
columns_to_check = ['LoanPeriod', 'Renewals', 'RenewalPeriod', 'TimeUnit']

for index, row in df_dangtailieu.iterrows():
    for col in columns_to_check:
        if pd.isna(row[col]):
            df_dangtailieu.at[index, col] = 0  # Thay NaN bằng 0
            
print(df_dangtailieu)

    Dang_tai_lieu_ID                                    Dang_tai_lieu  \
0                  1                     Sách, chuyên khảo, tuyển tập   
1                  2                                        Bài trích   
2                  3                                Luận án, luận văn   
3                  4  Báo cáo kết quả  nghiên cứu; tổng kết; khảo sát   
4                  5                                 Báo cáo hội nghị   
5                  6                               Catalô công nghiệp   
6                  7                                       Tiêu chuẩn   
7                  8                                         Sáng chế   
8                  9                                  ấn phẩm định kỳ   
9                 10                                             Phim   
10                11                              Bản đồ, sách bản đồ   
11                12                    Hình vẽ, bản vẽ, tranh ảnh,..   
12                13                     Tờ rời giớ

## Load data

In [76]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Dang_tai_lieu (ID_dang_tai_lieu, Dang_tai_lieu, Ky_hieu_tai_lieu, LoanPeriod, Renewals, RenewalPeriod, TimeUnit, Fee, OverdueFine, FixedFee) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
for index, row in df_dangtailieu.iterrows():
    values = (row['Dang_tai_lieu_ID'], 
                row['Dang_tai_lieu'],
                row['Ky_hieu_tai_lieu'],
                row['LoanPeriod'],
                row['Renewals'],
                row['RenewalPeriod'],
                row['TimeUnit'],
                row['Fee'],
                row['OverdueFine'],
                row['FixedFee'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Ten_form

## Xóa data Dim_Ten_form

In [77]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Ten_form"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [78]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Tenform = """SELECT ID, 
                        dbo.DecodeUTF8String(Ten_form) AS Ten_form, 
                        Nguoi_tao,
                        Ngay_tao,
                        Ngay_sua_cuoi
                        FROM Ten_Form
                        """
df_tenform = pd.read_sql(query_Tenform, conn_libol)
print(df_tenform)

    ID                         Ten_form      Nguoi_tao            Ngay_tao  \
0   37                    Sách (USMARC)  Administrator 2001-05-19 16:00:51   
1   38              Băng Video (USMARC)  Administrator 2001-05-31 16:58:22   
2   40                Âm thanh (USMARC)  Administrator 2001-06-02 08:52:53   
3   39            Tệp máy tính (USMARC)  Administrator 2001-05-31 17:34:27   
4   41                  Bản đồ (USMARC)  Administrator 2001-08-09 17:20:27   
5   42         Ấn phẩm định kỳ (USMARC)  Administrator 2001-08-10 08:46:06   
6   43                   Sách (rút gọn)  Administrator 2001-08-21 11:50:25   
7   81  Biên mục Tiêu chuẩn và Quy phạm         DHSPKT 2002-08-21 08:55:07   
8   67                  Bài báo(DHSPKT)  Administrator 2001-11-27 11:09:01   
9   68                        Bài trích  Administrator 2001-11-27 11:16:28   
10  85               Biên mục Bài trích  Administrator 2002-08-21 12:36:16   
11  82                    Biên mục Sách         DHSPKT 2002-08-2

C:\Users\phung\AppData\Local\Temp\ipykernel_24616\64601752.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tenform = pd.read_sql(query_Tenform, conn_libol)


## Xử lý data

In [79]:
df_tenform['Ngay_tao'] = df_tenform['Ngay_tao'].dt.strftime('%Y%m%d').astype(int)
df_tenform['Ngay_sua_cuoi'] = df_tenform['Ngay_sua_cuoi'].dt.strftime('%Y%m%d').astype(int)
print(df_tenform)

    ID                         Ten_form      Nguoi_tao  Ngay_tao  \
0   37                    Sách (USMARC)  Administrator  20010519   
1   38              Băng Video (USMARC)  Administrator  20010531   
2   40                Âm thanh (USMARC)  Administrator  20010602   
3   39            Tệp máy tính (USMARC)  Administrator  20010531   
4   41                  Bản đồ (USMARC)  Administrator  20010809   
5   42         Ấn phẩm định kỳ (USMARC)  Administrator  20010810   
6   43                   Sách (rút gọn)  Administrator  20010821   
7   81  Biên mục Tiêu chuẩn và Quy phạm         DHSPKT  20020821   
8   67                  Bài báo(DHSPKT)  Administrator  20011127   
9   68                        Bài trích  Administrator  20011127   
10  85               Biên mục Bài trích  Administrator  20020821   
11  82                    Biên mục Sách         DHSPKT  20020821   
12  83          Biên mục Báo và Tạp chí         DHSPKT  20020821   
13  84                 Biên mục Luận án         

## Load data

In [80]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Ten_form (ID_form, Ten_form, Nguoi_tao, Ngay_tao, Ngay_sua_cuoi) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_tenform.iterrows():
    values = (row['ID'], 
                row['Ten_form'],
                row['Nguoi_tao'],
                row['Ngay_tao'],
                row['Ngay_sua_cuoi'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng DIM_Thu_vien

## Xóa data bảng Dim_Thu_vien

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Thu_vien"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server 

In [83]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Thuvien = """SELECT dbo.DecodeUTF8String(Ten_viet_tat) AS Ten_viet_tat,
                        dbo.DecodeUTF8String(Thu_vien) AS Thu_vien, 
                        dbo.DecodeUTF8String(Dia_chi) AS Dia_chi,
                        gia_tri,
                        LocalLib
                        FROM Thu_vien
                        """
df_thuvien = pd.read_sql(query_Thuvien, conn_libol)
print(df_thuvien)

                    Ten_viet_tat                            Thu_vien  \
0                         DHSPKT   Thư viện Đại học Sư Phạm Kĩ Thuật   
1                         ĐHSPKT                                None   
2                        SDHSPKT                                None   
3                    ĐHSPKT##Vie                                None   
4                            Vie                                None   
5                    DHSPKT##Vie                                None   
6                    D9HSP T.HCM                                None   
7                         SPDHKT                                None   
8                  ĐHSPKT TP.HCM                                None   
9                          ĐSPKT                                None   
10                        HCMUTE                                None   
11                           DLC                                None   
12  TVTTHCM|bvie|cTVTTHCM|eAACR2                                

C:\Users\phung\AppData\Local\Temp\ipykernel_24616\133294711.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_thuvien = pd.read_sql(query_Thuvien, conn_libol)


## Load data

In [85]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Thu_vien (ID_thu_vien, Thu_vien, Dia_chi, Gia_tri, LocalLib) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_thuvien.iterrows():
    values = (row['Ten_viet_tat'], 
                row['Thu_vien'],
                row['Dia_chi'],
                row['gia_tri'],
                row['LocalLib'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Kho

## Xoá data trong bảng Dim_Kho

In [99]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Kho"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [95]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Kho= """SELECT Kho_ID, dbo.DecodeUTF8String(Kho) AS Kho, Thu_vien_ID, MaxID, Mo FROM Kho """
df_kho = pd.read_sql(query_Kho, conn_libol)

query_Thuvien = """SELECT Thu_vien_ID, dbo.DecodeUTF8String(Ten_viet_tat) AS Ten_viet_tat FROM Thu_vien """
df_thuvien = pd.read_sql(query_Thuvien, conn_libol)

print(df_kho)
print(df_thuvien)

   Kho_ID           Kho  Thu_vien_ID  MaxID     Mo
0       7            KM            2      2   True
1       8            NV            1  71094   True
2       9            GT            1  60109   True
3       5            KM            1  14805   True
4       6            KD            1  10574  False
5      19   Đọc tại chỗ            1      1  False
6      13  Kho thanh lý            1      1  False
7      14   Unavailbale            1      1  False
8      16           CLC            1    104  False
9      18       Kho Lưu            1      4  False
    Thu_vien_ID                  Ten_viet_tat
0             1                        DHSPKT
1             2                        ĐHSPKT
2             3                       SDHSPKT
3             4                   ĐHSPKT##Vie
4             5                           Vie
5             6                   DHSPKT##Vie
6             7                   D9HSP T.HCM
7             8                        SPDHKT
8             9          

C:\Users\phung\AppData\Local\Temp\ipykernel_24616\2520694761.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_kho = pd.read_sql(query_Kho, conn_libol)
C:\Users\phung\AppData\Local\Temp\ipykernel_24616\2520694761.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_thuvien = pd.read_sql(query_Thuvien, conn_libol)


## Xử lý data

In [96]:
map_dict = dict(zip(df_thuvien['Thu_vien_ID'], df_thuvien['Ten_viet_tat']))

for index, row in df_kho.iterrows():
    id_thu_vien = row['Thu_vien_ID']
    if id_thu_vien in map_dict:
        # Thay thế ID_Thu_vien bằng Ten_viet_tat nếu khớp
        df_kho.at[index, 'Thu_vien_ID'] = map_dict[id_thu_vien]

print(df_kho)


   Kho_ID           Kho Thu_vien_ID  MaxID     Mo
0       7            KM      ĐHSPKT      2   True
1       8            NV      DHSPKT  71094   True
2       9            GT      DHSPKT  60109   True
3       5            KM      DHSPKT  14805   True
4       6            KD      DHSPKT  10574  False
5      19   Đọc tại chỗ      DHSPKT      1  False
6      13  Kho thanh lý      DHSPKT      1  False
7      14   Unavailbale      DHSPKT      1  False
8      16           CLC      DHSPKT    104  False
9      18       Kho Lưu      DHSPKT      4  False


C:\Users\phung\AppData\Local\Temp\ipykernel_24616\2981671625.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ĐHSPKT' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_kho.at[index, 'Thu_vien_ID'] = map_dict[id_thu_vien]


## Load data

In [100]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Kho (ID_kho, Kho, ID_thu_vien, MaxID, Mo) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_kho.iterrows():
    values = (row['Kho_ID'], 
                row['Kho'],
                row['Thu_vien_ID'],
                row['MaxID'],
                row['Mo'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Bim_Ban_doc

## Xoá data bảng Dim_Ban_doc

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Ban_doc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [41]:
query_Bandoc = """
SELECT TOP (100) So_the,
       dbo.DecodeUTF8String(Ho_ten) AS Ho_ten,
       Ngay_sinh,
       Dan_toc_ID,
       Trinh_do_ID,
       dbo.DecodeUTF8String(So_dien_thoai) AS So_dien_thoai,
       dbo.DecodeUTF8String(Nghe_nghiep) AS Nghe_nghiep,
       dbo.DecodeUTF8String(Co_quan) AS Co_quan,
       dbo.DecodeUTF8String(chuc_vu) AS chuc_vu,
       dbo.DecodeUTF8String(Dia_chi) AS Dia_chi,
       dbo.DecodeUTF8String(Dia_chi_thuong_tru) AS Dia_chi_thuong_tru,
       ID_Khoa_hoc,
       Lop,
       Anh,
       Ngay_cap,
       Ngay_het_han,
       Email,
       Nhom_ID,
       Nhom_nghanh_nghe_ID,
       Gioi_tinh,
       Status,
       dbo.DecodeUTF8String(Ghi_chu) AS Ghi_chu,
       Mat_khau
  FROM Ban_doc
"""
df_bandoc = pd.read_sql(query_Bandoc, conn_libol)
print(df_bandoc)


C:\Users\phung\AppData\Local\Temp\ipykernel_27428\1563414921.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bandoc = pd.read_sql(query_Bandoc, conn_libol)


      So_the              Ho_ten  Ngay_sinh  Dan_toc_ID  Trinh_do_ID  \
0   98101150      Huỳnh Quốc Bảo 1980-03-12           1            9   
1   98101153     Phan Thanh Bình 1980-12-23           1            9   
2   98101154   Ngô Đình Sơn Cước 1980-07-31           1            9   
3   98101158      Lê Vũ Công Dân        NaT           1            9   
4   SKT00003       NGUYỄN VĂN VỊ 1966-03-12           1            3   
..       ...                 ...        ...         ...          ...   
95  01710020         LÊ THANH HÀ        NaT           1            9   
96  01710011     DƯƠNG MINH DANH        NaT           1            9   
97  01105014   ĐỖ PH. THÀNH DŨNG 1982-08-25           1            9   
98  00108087  TRẦN T. THANH NHÀN        NaT           1            9   
99  00107033     ĐINH MINH HOÀNG        NaT           1            9   

   So_dien_thoai Nghe_nghiep        Co_quan chuc_vu  \
0                  Sinh viên  Trường ĐHSPKT    None   
1                  Sinh v

## Xử lý data

### Xử lý kiểu date

In [42]:
#chuyển những dòng kiểu date thành int để map với datekey
df_bandoc['Ngay_sinh'] = df_bandoc['Ngay_sinh'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) else 0)
df_bandoc['Ngay_cap'] = df_bandoc['Ngay_cap'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) else 0)
df_bandoc['Ngay_het_han'] = df_bandoc['Ngay_het_han'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) else 0)

print(df_bandoc)


      So_the              Ho_ten  Ngay_sinh  Dan_toc_ID  Trinh_do_ID  \
0   98101150      Huỳnh Quốc Bảo   19800312           1            9   
1   98101153     Phan Thanh Bình   19801223           1            9   
2   98101154   Ngô Đình Sơn Cước   19800731           1            9   
3   98101158      Lê Vũ Công Dân          0           1            9   
4   SKT00003       NGUYỄN VĂN VỊ   19660312           1            3   
..       ...                 ...        ...         ...          ...   
95  01710020         LÊ THANH HÀ          0           1            9   
96  01710011     DƯƠNG MINH DANH          0           1            9   
97  01105014   ĐỖ PH. THÀNH DŨNG   19820825           1            9   
98  00108087  TRẦN T. THANH NHÀN          0           1            9   
99  00107033     ĐINH MINH HOÀNG          0           1            9   

   So_dien_thoai Nghe_nghiep        Co_quan chuc_vu  \
0                  Sinh viên  Trường ĐHSPKT    None   
1                  Sinh v

### Xử lý Dan_toc

In [43]:
# đọc dữ liệu lấy từ bộ về
df_Data_Dim_Dan_toc = pd.read_csv("./Source-Data/Data_Dim_Dan_toc.csv")
df_Data_Dim_Dan_toc = df_Data_Dim_Dan_toc.where(pd.notnull(df_Data_Dim_Dan_toc), None)
# đọc dữ liệu đã lưu trong sql server
query_Dan_toc = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "
df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)

#tìm và mapping 2 bảng lại
df_mapping = df_Dan_toc.copy()
df_mapping['Mapping Mã'] = None
df_mapping['CSV Mã'] = df_Data_Dim_Dan_toc['Mã']
df_mapping['CSV Tên'] = df_Data_Dim_Dan_toc['Tên']
df_mapping['CSV Tên khác'] = df_Data_Dim_Dan_toc['Tên gọi khác'].str.lower()
for i, dan_toc in enumerate(df_mapping['Dan_toc']):
    dan_toc = dan_toc.lower()
    dan_toc_bogach = dan_toc.replace("-", " ")
    dan_toc_botrong = dan_toc.replace(" ", "-")

    for j, row in df_mapping.iterrows():
        CSV_ten = row['CSV Tên'].lower() if pd.notna(row['CSV Tên']) else ""
        CSV_ten_khac = row['CSV Tên khác'].lower() if pd.notna(row['CSV Tên khác']) else ""
        if ((pd.notna(CSV_ten) and dan_toc == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc in CSV_ten_khac) or 
            (pd.notna(CSV_ten) and dan_toc_bogach == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_bogach in CSV_ten_khac) or
            (pd.notna(CSV_ten) and dan_toc_botrong == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_botrong in CSV_ten_khac)):
            df_mapping.at[i, 'Mapping Mã'] = row['CSV Mã']
            break
        else:
            df_mapping.at[i, 'Mapping Mã'] = 56
print(df_mapping)

C:\Users\phung\AppData\Local\Temp\ipykernel_27428\3819844708.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)


    Id  Dan_toc Mapping Mã  CSV Mã CSV Tên  \
0    1     Kinh        1.0     1.0    Kinh   
1    2    Mường        3.0     2.0     Tày   
2    3      Tày        2.0     3.0    Thái   
3    4     Thái        3.0     4.0     Hoa   
4    5      Hoa        4.0     5.0  Khơ-me   
..  ..      ...        ...     ...     ...   
68  71     Ê Đê       12.0     NaN     NaN   
69  72      Thổ        2.0     NaN     NaN   
70  73    Kờ Ho         56     NaN     NaN   
71  74     Jrai         56     NaN     NaN   
72  75  Châu mạ       28.0     NaN     NaN   

                                         CSV Tên khác  
0                                                việt  
1           thổ, ngạn, phén, thù lao, pa dí, tày khao  
2   tày đăm, tày mười, tày thanh, mán thanh, hàng ...  
3   hán, triều châu, phúc kiến, quảng đông, hải na...  
4              cur, cul, cu, thổ, việt gốc miên, krôm  
..                                                ...  
68                                                NaN  

In [49]:
map_dict = dict(zip(df_mapping['Id'], df_mapping['Mapping Mã'])) # Tạo map_dict để ánh xạ từ Id sang Mapping_Ma trong df_mapping

for index, row in df_bandoc.iterrows(): # Lặp qua từng dòng trong df_ban_doc để cập nhật Dan_toc_ID
    dan_toc_id = row['Dan_toc_ID']
    
    if pd.isna(dan_toc_id): # Kiểm tra nếu Dan_toc_ID rỗng (None hoặc NaN)
        df_bandoc.at[index, 'Dan_toc_ID'] = 0
    else:
        if dan_toc_id in map_dict: # Kiểm tra nếu Dan_toc_ID có trong map_dict
            df_bandoc.at[index, 'Dan_toc_ID'] = map_dict[dan_toc_id]# Nếu tìm thấy, thay thế bằng giá trị Mapping_Ma
        else:
            df_bandoc.at[index, 'Dan_toc_ID'] = 0 # Nếu không tìm thấy, gán Dan_toc_ID bằng 0
print(df_bandoc)


      So_the              Ho_ten  Ngay_sinh  Dan_toc_ID  Trinh_do_ID  \
0   98101150      Huỳnh Quốc Bảo   19800312           1            9   
1   98101153     Phan Thanh Bình   19801223           1            9   
2   98101154   Ngô Đình Sơn Cước   19800731           1            9   
3   98101158      Lê Vũ Công Dân          0           1            9   
4   SKT00003       NGUYỄN VĂN VỊ   19660312           1            3   
..       ...                 ...        ...         ...          ...   
95  01710020         LÊ THANH HÀ          0           1            9   
96  01710011     DƯƠNG MINH DANH          0           1            9   
97  01105014   ĐỖ PH. THÀNH DŨNG   19820825           1            9   
98  00108087  TRẦN T. THANH NHÀN          0           1            9   
99  00107033     ĐINH MINH HOÀNG          0           1            9   

   So_dien_thoai Nghe_nghiep        Co_quan chuc_vu  \
0                  Sinh viên  Trường ĐHSPKT    None   
1                  Sinh v

### Xử lý Lop


In [60]:
query_lop = "SELECT * FROM DIM_Lop"
df_lop = pd.read_sql(query_lop, conn_dwh_lib)
print(df_lop)

     ID_lop   Ten_lop ID_khoa
0         0  Không rõ       0
1         1    001011       0
2        10    001021       0
3       100   014010A       0
4      1000   087030B       0
...     ...       ...     ...
4256    995   087011A       0
4257    996   087020A       0
4258    997   087020B       0
4259    998   087020C       0
4260    999   087030A       0

[4261 rows x 3 columns]


C:\Users\phung\AppData\Local\Temp\ipykernel_27428\3524043409.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop = pd.read_sql(query_lop, conn_dwh_lib)


In [ ]:
query_lop = "SELECT * FROM DIM_Lop"
df_lop = pd.read_sql(query_lop, conn_dwh_lib)

map_dict = dict(zip(df_lop['Ten_lop'], df_lop['Ten_lop'])) # Tạo map_dict để ánh xạ 

for index, row in df_bandoc.iterrows(): # Lặp qua từng dòng trong df_ban_doc để cập nhật lop_id
    lop_id = row['Lop']
    
    if pd.isna(lop_id): # Kiểm tra nếu lop_id rỗng (None hoặc NaN)
        df_bandoc.at[index, 'Lop'] = 0
    else:
        if lop_id in map_dict: # Kiểm tra nếu lop_id có trong map_dict
            df_bandoc.at[index, 'Lop'] = map_dict[lop_id]# Nếu tìm thấy, thay thế bằng giá trị Mapping_Ma
        else:
            df_bandoc.at[index, 'Dan_toc_ID'] = 0 # Nếu không tìm thấy, gán lop_id bằng 0

print(df_bandoc)

{'0': 'Không rõ', '1': '001011', '10': '001021', '100': '014010A', '1000': '087030B', '1001': '087030C', '1002': '087031A', '1003': '087031C', '1004': '08703A', '1005': '08703B1', '1006': '087050A', '1007': '087050B', '1008': '087050C', '1009': '087090A', '101': '01401CA', '1010': '087090B', '1011': '087090C', '1012': '08902LD1', '1013': '08902LD2', '1014': '08902LĐ2', '1015': '089030A', '1016': '08917LD1', '1017': '08917LD2', '1018': '08917LD3', '1019': '08BM', '102': '01402PY', '1020': '08D02VDA', '1021': '08D02VDB', '1022': '08D02VDC', '1023': '08D02VDD', '1024': '08D02VDE', '1025': '08D03VDA', '1026': '08D03VDB', '1027': '08D03VDC', '1028': '08D03VDD', '1029': '08D03VDE', '103': '01402TV', '1030': '08ED', '1031': '08FB', '1032': '091011A', '1033': '091011B', '1034': '091011C', '1035': '091012A', '1036': '091012B', '1037': '091012C', '1038': '09101CL1', '1039': '09101CL2', '104': '01404CA', '1040': '091021A', '1041': '091021B', '1042': '091021C', '1043': '091022A', '1044': '091022B'

C:\Users\phung\AppData\Local\Temp\ipykernel_27428\2072791990.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop = pd.read_sql(query_lop, conn_dwh_lib)
